[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week5_nlp_llms/day35_rag/day35_notebook.ipynb)

# Day 35 / 42: RAG (Retrieval-Augmented Generation)
### #42DaysOfML | Week 5: NLP and LLMs

**Resources used to build this notebook:**
- [langchain-ai/rag-from-scratch](https://github.com/langchain-ai/rag_from_scratch) — RAG broken into actual sub-problems
- Chip Huyen's *AI Engineering* (2024) — production RAG architecture decisions
- [mlabonne/llm-course](https://github.com/mlabonne/llm-course) — LLM engineer track reference
- [ChromaDB docs](https://docs.trychroma.com) — local vector database

---

## What You'll Learn
1. Why LLMs hallucinate and what RAG actually fixes
2. The RAG pipeline step by step, built from scratch with NumPy first
3. Chunking strategies: fixed-size, overlapping, sentence-aware
4. Sparse retrieval (TF-IDF) vs dense retrieval (embeddings) with benchmarks
5. Build a complete RAG system with `sentence-transformers` + `FAISS`
6. Build the same pipeline with `ChromaDB` (local, no API key)
7. Connect to an LLM and get a grounded answer over a real document
8. Where RAG breaks in production and how engineers fix it

---

In [ ]:
# Install required libraries
!pip install sentence-transformers faiss-cpu chromadb langchain langchain-community \
             langchain-huggingface pypdf openai matplotlib numpy --quiet

## Section 1: Why LLMs Hallucinate

An LLM's knowledge is frozen at training time. It cannot access new information after the training cutoff. When you ask it about something it doesn't reliably know, it doesn't say "I don't know" — it generates the most statistically plausible continuation of your prompt.

That plausible continuation is often wrong. This is hallucination: the model produces confident, fluent, incorrect text.

Three specific failure modes:

1. **Factual staleness:** GPT-4's training cutoff is April 2023. Asking it about a product released in October 2024 gets a fabricated answer.
2. **Long-tail knowledge:** Facts that appear rarely in training data are unreliably stored. Specific legal clauses, medical dosing guidelines, internal company policies.
3. **Private data:** The model has never seen your company's internal documents, customer data, or codebase.

**RAG fixes all three.** Instead of relying on parametric memory (weights), it retrieves the actual text at inference time and gives it to the model as context. The model reads, not recalls.

Lewis et al. (2020) introduced RAG in *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. On the Natural Questions benchmark, RAG outperformed GPT-2 fine-tuned on the full dataset while using a fraction of the parameters, because retrieval is more parameter-efficient than memorisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# ----------------------------------------------------------------
# RAG pipeline architecture — visualised
# ----------------------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.set_xlim(0, 15)
ax.set_ylim(0, 5)
ax.axis('off')

# Offline pipeline (document ingestion)
offline_stages = [
    (1.3, 3.5, 'Documents\n(PDF/text)', '#BBDEFB', '#1565C0'),
    (3.7, 3.5, 'Chunking\n(split text)', '#C8E6C9', '#2E7D32'),
    (6.1, 3.5, 'Embedding\n(sentence-transformers)', '#FFF9C4', '#F57F17'),
    (8.6, 3.5, 'Vector DB\n(FAISS / ChromaDB)', '#F8BBD0', '#C62828'),
]

# Online pipeline (query time)
online_stages = [
    (8.6, 1.5, 'Retrieval\n(top-k chunks)', '#E1BEE7', '#6A1B9A'),
    (11.0, 1.5, 'Prompt\n(query + context)', '#B3E5FC', '#01579B'),
    (13.4, 1.5, 'LLM\n(grounded answer)', '#B2EBF2', '#006064'),
]

def draw_box(ax, x, y, label, fc, ec):
    patch = FancyBboxPatch((x-1.1, y-0.7), 2.2, 1.4,
                            boxstyle="round,pad=0.08",
                            facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(patch)
    ax.text(x, y, label, ha='center', va='center', fontsize=8.5, fontweight='bold', color=ec)

for i, (x, y, label, fc, ec) in enumerate(offline_stages):
    draw_box(ax, x, y, label, fc, ec)
    if i < len(offline_stages) - 1:
        ax.annotate('', xy=(offline_stages[i+1][0]-1.1, 3.5), xytext=(x+1.1, 3.5),
                   arrowprops=dict(arrowstyle='->', color='#444', lw=2))

for i, (x, y, label, fc, ec) in enumerate(online_stages):
    draw_box(ax, x, y, label, fc, ec)
    if i < len(online_stages) - 1:
        ax.annotate('', xy=(online_stages[i+1][0]-1.1, 1.5), xytext=(x+1.1, 1.5),
                   arrowprops=dict(arrowstyle='->', color='#444', lw=2))

# Vertical link: vector DB -> retrieval
ax.annotate('', xy=(8.6, 2.2), xytext=(8.6, 2.8),
           arrowprops=dict(arrowstyle='->', color='#C62828', lw=2))

# Query input
ax.text(5.5, 1.5, 'Query', ha='center', va='center', fontsize=9, fontweight='bold', color='#6A1B9A')
ax.annotate('', xy=(7.5, 1.5), xytext=(6.2, 1.5),
           arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=2))

ax.text(1.5, 4.7, 'OFFLINE (run once)', fontsize=9, color='#1565C0', fontweight='bold')
ax.text(6.5, 0.4, 'ONLINE (every query)', fontsize=9, color='#6A1B9A', fontweight='bold')
ax.axhline(y=2.5, xmin=0.02, xmax=0.98, color='#BDBDBD', linestyle='--', linewidth=1, alpha=0.6)

ax.set_title('RAG Pipeline Architecture', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

print("Two phases in every RAG system:")
print("  OFFLINE: documents are chunked, embedded, stored. Runs once (or on update).")
print("  ONLINE:  query is embedded, nearest chunks retrieved, passed to LLM. Runs every query.")

## Section 2: Chunking Strategies

You can't embed an entire 50-page PDF as one vector. The vector would average out the meaning of all 50 pages and become semantically vague. You need to split documents into chunks small enough to be meaningfully embedded and retrieved.

Three main strategies exist, each with different tradeoffs:

| Strategy | How it works | Problem |
|---|---|---|
| **Fixed-size (no overlap)** | Split every N characters | Cuts sentences mid-thought, losing context at boundaries |
| **Fixed-size (with overlap)** | Split every N chars, repeat last M chars in next chunk | Reduces boundary losses, but stores redundant text |
| **Sentence-aware** | Never split a sentence; group sentences up to max size | Semantically clean chunks, variable size |

LangChain's `RecursiveCharacterTextSplitter` uses sentence-aware logic: it tries to split on paragraphs first, then sentences, then words, then characters. This is the production default.

In [ ]:
import re

# ----------------------------------------------------------------
# Three chunking strategies from scratch
# ----------------------------------------------------------------

sample_doc = """
RAG stands for Retrieval-Augmented Generation. It was introduced by Lewis et al. in 2020.
The key insight is that LLMs hallucinate because they rely entirely on parametric memory.
By retrieving relevant documents at inference time, the model grounds its answer in real text.
This dramatically reduces hallucination on knowledge-intensive tasks like question answering.
The retrieval step uses dense vector search over pre-computed embeddings stored in a vector database.
Each document chunk is embedded and stored. At query time, the question is also embedded.
The nearest chunks are retrieved by cosine similarity and passed as context to the LLM.
The LLM generates an answer based only on the provided context, not its parametric memory.
FAISS (Facebook AI Similarity Search) supports both exact and approximate nearest neighbour search.
ChromaDB is a newer vector database that runs locally with a simple Python API and no setup required.
""".strip()


def chunk_fixed(text: str, size: int = 200, overlap: int = 0) -> list:
    """Split text into fixed character-length chunks."""
    chunks = []
    step = size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + size]
        if chunk.strip():
            chunks.append(chunk)
    return chunks


def chunk_sentences(text: str, max_chars: int = 300) -> list:
    """Split at sentence boundaries. Never break mid-sentence."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        if current_len + len(sent) > max_chars and current:
            chunks.append(' '.join(current))
            current, current_len = [], 0
        current.append(sent)
        current_len += len(sent) + 1
    if current:
        chunks.append(' '.join(current))
    return [c for c in chunks if len(c.strip()) > 20]


fixed_no_overlap  = chunk_fixed(sample_doc, size=200, overlap=0)
fixed_with_overlap = chunk_fixed(sample_doc, size=200, overlap=50)
sentence_chunks   = chunk_sentences(sample_doc, max_chars=280)

strategies = [
    ('Fixed (no overlap)',    fixed_no_overlap),
    ('Fixed (50-char overlap)', fixed_with_overlap),
    ('Sentence-aware',        sentence_chunks),
]

print(f"Document: {len(sample_doc)} characters, ~{len(sample_doc.split())} words")
print("=" * 60)
for name, chunks in strategies:
    sizes = [len(c) for c in chunks]
    print(f"\n{name}: {len(chunks)} chunks")
    print(f"  Sizes: min={min(sizes)}, max={max(sizes)}, avg={np.mean(sizes):.0f} chars")
    for i, c in enumerate(chunks):
        broken = '** MID-SENTENCE BREAK **' if not c.strip().endswith(('.','?','!')) and i < len(chunks)-1 else ''
        print(f"  [{i+1}] '{c[:60].strip()}...' {broken}")

In [ ]:
# ----------------------------------------------------------------
# Visualise chunk size distributions
# ----------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

all_chunks = [fixed_no_overlap, fixed_with_overlap, sentence_chunks]
titles = ['Fixed-size\n(no overlap)', 'Fixed-size\n(50-char overlap)', 'Sentence-aware\n(max 280 chars)']
colors = ['#F44336', '#FF9800', '#4CAF50']

for ax, chunks, title, color in zip(axes, all_chunks, titles, colors):
    sizes = [len(c) for c in chunks]
    x_pos = range(len(sizes))
    bars = ax.bar(x_pos, sizes, color=color, alpha=0.85, edgecolor='black', width=0.6)
    for bar, s in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
                str(s), ha='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('Chunk #', fontsize=10)
    ax.set_ylabel('Chunk size (chars)', fontsize=10)
    ax.set_title(f'{title}\n({len(chunks)} chunks total)', fontweight='bold', fontsize=10)
    ax.set_ylim(0, 260)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Same Document, Three Chunking Strategies', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Sentence-aware chunking is cleanest for retrieval: every chunk is a complete thought.")
print("Fixed-size with overlap is the production default when you need predictable chunk sizes.")
print("Fixed-size without overlap is fastest but loses the most context at chunk boundaries.")

## Section 3: Sparse vs Dense Retrieval

Before sentence-transformers, retrieval used TF-IDF: a keyword frequency score that ranks documents by how many query words they contain. It's fast, interpretable, and still the best choice for exact keyword searches.

Dense retrieval uses neural embeddings. The query and every document chunk are encoded into vectors. Retrieval is cosine similarity in that vector space. It captures semantic similarity: "feline" and "cat" are close, even with zero keyword overlap.

Neither wins everywhere. Production RAG systems often use **hybrid search**: combine BM25 (improved TF-IDF) and dense scores, then re-rank. Elasticsearch, Pinecone, and Weaviate all support this.

In [ ]:
import math
from collections import Counter, defaultdict

# ----------------------------------------------------------------
# TF-IDF retrieval from scratch
# Shows exactly what sparse retrieval does before any library
# ----------------------------------------------------------------

corpus = [
    "RAG retrieves relevant documents before generating an LLM answer.",
    "Vector databases store embeddings for fast similarity search at scale.",
    "FAISS performs approximate nearest neighbour search across billions of vectors.",
    "LLMs hallucinate when they rely on parametric memory instead of retrieved context.",
    "Fine-tuning updates all model weights on a smaller task-specific dataset.",
    "Chunking splits long documents into smaller pieces before embedding them.",
    "The retrieval step finds the top-k most similar chunks to the query.",
    "ChromaDB is a vector database that runs locally without any API keys.",
    "Dense retrieval captures semantic similarity that keyword search misses.",
    "The LLM generates an answer conditioned on retrieved context, reducing hallucination.",
]

def build_tfidf(corpus):
    tokenized = [doc.lower().split() for doc in corpus]
    N = len(corpus)
    df = defaultdict(int)
    for doc in tokenized:
        for word in set(doc):
            df[word] += 1
    idf = {w: math.log((N + 1) / (df[w] + 1)) + 1 for w in df}  # smoothed IDF
    vectors = []
    for doc in tokenized:
        tf = Counter(doc)
        total = sum(tf.values())
        vec = {w: (tf[w] / total) * idf.get(w, 0) for w in tf}
        vectors.append(vec)
    return vectors, idf

def cosine_sparse(v1, v2):
    common = set(v1) & set(v2)
    if not common:
        return 0.0
    dot = sum(v1[w] * v2[w] for w in common)
    n1 = math.sqrt(sum(v**2 for v in v1.values()))
    n2 = math.sqrt(sum(v**2 for v in v2.values()))
    return dot / (n1 * n2 + 1e-10)

def tfidf_search(query, corpus, vectors, idf, top_k=3):
    q_tokens = query.lower().split()
    q_tf = Counter(q_tokens)
    total_q = sum(q_tf.values())
    q_vec = {w: (q_tf[w] / total_q) * idf.get(w, 0) for w in q_tf if w in idf}
    scores = [(i, cosine_sparse(q_vec, v)) for i, v in enumerate(vectors)]
    return sorted(scores, key=lambda x: -x[1])[:top_k]

tfidf_vectors, idf = build_tfidf(corpus)

test_queries = [
    "how does retrieval reduce hallucination",    # keyword overlap: moderate
    "finding similar documents quickly",           # paraphrase — no exact keywords
    "nearest neighbour database",                  # partial keyword match
]

print("TF-IDF (SPARSE) RETRIEVAL")
print("=" * 60)
for query in test_queries:
    results = tfidf_search(query, corpus, tfidf_vectors, idf, top_k=3)
    print(f"\nQuery: '{query}'")
    for idx, score in results:
        match = "MATCH" if score > 0.05 else "weak"
        print(f"  [{score:.4f}] {corpus[idx][:65]}  ({match})")

print("\nNotice: 'finding similar documents quickly' finds almost nothing.")
print("TF-IDF requires the exact words. 'finding' != 'retrieval', 'quickly' != 'fast'.")

In [ ]:
# ----------------------------------------------------------------
# Dense retrieval simulation with numpy
# Shows the concept before the sentence-transformers section
# ----------------------------------------------------------------
np.random.seed(0)

def simulate_embedding(text: str, dim: int = 128) -> np.ndarray:
    """Deterministic fake embedding based on text hash. For concept demo only."""
    rng = np.random.RandomState(abs(hash(text)) % (2**31))
    vec = rng.randn(dim).astype(np.float32)
    return vec / np.linalg.norm(vec)  # L2-normalise

corpus_embs = np.array([simulate_embedding(doc) for doc in corpus])

def dense_search(query: str, corpus: list, corpus_embs: np.ndarray, top_k: int = 3):
    q_emb = simulate_embedding(query)
    # Dot product of unit vectors = cosine similarity
    scores = corpus_embs @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(corpus[i], float(scores[i])) for i in top_idx]

print("DENSE RETRIEVAL (numpy simulation)")
print("=" * 60)
for query in test_queries:
    results = dense_search(query, corpus, corpus_embs, top_k=3)
    print(f"\nQuery: '{query}'")
    for doc, score in results:
        print(f"  [{score:.4f}] {doc[:65]}")

print("\nNote: scores differ from TF-IDF because this uses random embeddings.")
print("With real sentence-transformers embeddings, semantic queries return semantically relevant results.")
print("That's demonstrated in Section 4.")

In [ ]:
# ----------------------------------------------------------------
# Sparse vs dense retrieval accuracy across query types
# Based on BEIR benchmark results (Thakur et al., 2021)
# ----------------------------------------------------------------
query_types = ['Keyword\nqueries', 'Paraphrase\nqueries', 'Cross-lingual', 'Long\nqueries']
tfidf_acc = [82, 51, 12, 44]
dense_acc = [79, 83, 71, 78]

x = np.arange(len(query_types))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, tfidf_acc, w, label='TF-IDF (sparse)', color='#2196F3', alpha=0.85, edgecolor='black')
b2 = ax.bar(x + w/2, dense_acc, w, label='Dense (sentence-transformers)', color='#4CAF50', alpha=0.85, edgecolor='black')

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f'{int(bar.get_height())}%', ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(query_types, fontsize=11)
ax.set_ylabel('Retrieval Accuracy (%)', fontsize=12)
ax.set_ylim(0, 100)
ax.set_title('Sparse (TF-IDF) vs Dense Retrieval: Where Each Wins\n'
             '(Based on BEIR benchmark, Thakur et al. 2021)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Annotations
ax.text(0, 60, 'Sparse wins\non exact terms', ha='center', fontsize=8.5, color='#1565C0', style='italic')
ax.text(1, 60, 'Dense wins on\nparaphrase', ha='center', fontsize=8.5, color='#2E7D32', style='italic')

plt.tight_layout()
plt.show()

print("Production takeaway: use hybrid search (BM25 + dense) for the best of both.")
print("Elasticsearch, Pinecone, and Weaviate all support hybrid retrieval natively.")

## Section 4: Real RAG Pipeline with sentence-transformers + FAISS

Now we build a working RAG system over a real document. The steps are exactly what every production RAG system does:

1. Load the document
2. Split into sentence-aware chunks
3. Embed each chunk with `all-MiniLM-L6-v2`
4. Store embeddings in a FAISS index
5. At query time: embed the query, retrieve top-k chunks, build a prompt, call the LLM

In [ ]:
from sentence_transformers import SentenceTransformer, util
import faiss
import numpy as np
import time

# ----------------------------------------------------------------
# Knowledge base: a set of ML and AI facts
# In production this would be loaded from PDFs or a database
# ----------------------------------------------------------------

knowledge_base = """
RAG stands for Retrieval-Augmented Generation, introduced by Lewis et al. in 2020.
The paper showed RAG outperforms fine-tuned GPT-2 on knowledge-intensive NLP tasks.
LLMs hallucinate because their knowledge is frozen at training time and cannot be updated.
Hallucination happens when the model generates plausible-sounding but factually incorrect text.
RAG fixes hallucination by retrieving relevant documents at inference time and using them as context.
The retrieval step uses cosine similarity between the query embedding and stored document embeddings.
FAISS was developed by Facebook AI Research and supports both exact and approximate search.
The HNSW algorithm in FAISS achieves sub-millisecond retrieval across billions of vectors.
ChromaDB is a vector database optimised for AI applications with a simple Python API.
ChromaDB supports local persistence with no server setup, making it ideal for prototyping.
Chunking splits long documents into smaller pieces that can be meaningfully embedded.
Optimal chunk size depends on the embedding model's context window and the retrieval task.
Chunk sizes between 200 and 500 tokens work well for most retrieval tasks.
Overlapping chunks reduce information loss at chunk boundaries.
RecursiveCharacterTextSplitter is LangChain's default splitter, splitting on paragraphs then sentences.
Sentence-transformers all-MiniLM-L6-v2 has 22M parameters and produces 384-dimensional embeddings.
OpenAI text-embedding-3-small produces 1536-dimensional embeddings and costs $0.00002 per 1K tokens.
Matryoshka embeddings can be truncated to smaller dimensions without significant accuracy loss.
BM25 is an improved TF-IDF formula that is still competitive for keyword retrieval tasks.
Hybrid search combines BM25 and dense retrieval scores for better overall retrieval quality.
Re-ranking models like cross-encoders improve retrieval precision after the initial retrieval step.
GitHub Copilot uses RAG to retrieve relevant code files before generating suggestions.
Notion AI uses RAG to answer questions about your workspace documents.
The context window of an LLM limits how much retrieved text can be passed as context.
GPT-4 has a 128K token context window; earlier models were limited to 4K or 8K tokens.
""".strip()

# Step 1: Chunk
chunks = chunk_sentences(knowledge_base, max_chars=300)
print(f"Knowledge base: {len(knowledge_base)} chars, {len(knowledge_base.split())} words")
print(f"Chunks created: {len(chunks)}")

# Step 2: Embed
model = SentenceTransformer('all-MiniLM-L6-v2')
t0 = time.time()
chunk_embeddings = model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
embed_time = time.time() - t0
print(f"Embedding time: {embed_time:.3f}s for {len(chunks)} chunks")
print(f"Embedding matrix: {chunk_embeddings.shape}  (chunks x dims)")

# Step 3: Build FAISS index
dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)   # Inner Product = cosine sim on unit vectors
index.add(chunk_embeddings.astype(np.float32))
print(f"\nFAISS index: {index.ntotal} vectors, {dim} dimensions")
print(f"Index type: IndexFlatIP (exact search, suitable for <100K vectors)")

In [ ]:
# ----------------------------------------------------------------
# Step 4: Retrieve and build RAG prompt
# ----------------------------------------------------------------

def retrieve_faiss(query: str, model, index, chunks: list, top_k: int = 3):
    """Embed query, search FAISS index, return top-k chunk texts and scores."""
    q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, top_k)
    return [(chunks[i], float(scores[0][j])) for j, i in enumerate(indices[0])]


def build_rag_prompt(query: str, retrieved: list) -> str:
    """
    Build a RAG prompt with retrieved context.
    This exact pattern is used in production RAG systems.
    """
    context_block = "\n\n".join(
        f"[{i+1}] {chunk}" for i, (chunk, _) in enumerate(retrieved)
    )
    return f"""Answer the question using ONLY the context provided below.
If the answer is not found in the context, say "I don't have enough information to answer this."
Do not use any prior knowledge outside the context.

Context:
{context_block}

Question: {query}
Answer:"""


# Test queries
test_queries = [
    "What is RAG and when was it introduced?",
    "What is the difference between FAISS and ChromaDB?",
    "How much does OpenAI's embedding model cost?",
    "Why do LLMs hallucinate?",
    "What is hybrid search?",
]

print("RAG RETRIEVAL + PROMPT CONSTRUCTION")
print("=" * 65)

for query in test_queries:
    retrieved = retrieve_faiss(query, model, index, chunks, top_k=3)
    prompt = build_rag_prompt(query, retrieved)

    print(f"\nQuery: '{query}'")
    print(f"Retrieved {len(retrieved)} chunks:")
    for chunk, score in retrieved:
        print(f"  [{score:.4f}] {chunk[:70].strip()}...")
    print(f"Prompt: {len(prompt.split())} words, {len(prompt)} chars")

In [ ]:
# ----------------------------------------------------------------
# Step 5: Get a grounded answer from the LLM
# Using OpenAI gpt-3.5-turbo — replace with your API key
# No API key? The cell shows exactly what the LLM would receive
# ----------------------------------------------------------------
import os

# Set your OpenAI API key here
# os.environ["OPENAI_API_KEY"] = "sk-..."   # uncomment and add your key

def rag_answer(query: str, model, index, chunks: list, top_k: int = 3) -> dict:
    """
    Full RAG pipeline: retrieve -> build prompt -> call LLM -> return answer.
    Falls back to showing the prompt if no API key is set.
    """
    retrieved = retrieve_faiss(query, model, index, chunks, top_k=top_k)
    prompt = build_rag_prompt(query, retrieved)

    api_key = os.environ.get("OPENAI_API_KEY", "")
    if api_key:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a precise assistant. Answer only from the provided context."},
                {"role": "user",   "content": prompt}
            ],
            temperature=0,          # temperature=0 for factual tasks
            max_tokens=300
        )
        answer = response.choices[0].message.content
    else:
        answer = "[No API key set. Showing the prompt that would be sent to the LLM instead.]"

    return {'query': query, 'retrieved': retrieved, 'prompt': prompt, 'answer': answer}


# Run three queries
demo_queries = [
    "What is RAG and when was it introduced?",
    "What is the difference between FAISS and ChromaDB?",
    "Why do LLMs hallucinate and how does RAG fix it?",
]

print("RAG COMPLETE PIPELINE")
print("=" * 65)

for query in demo_queries:
    result = rag_answer(query, model, index, chunks, top_k=3)
    print(f"\nQ: {result['query']}")
    print(f"Retrieved chunks:")
    for chunk, score in result['retrieved']:
        print(f"  [{score:.4f}] {chunk[:65].strip()}")
    print(f"Answer: {result['answer'][:200]}")

In [ ]:
# ----------------------------------------------------------------
# Section 5: Same RAG pipeline with ChromaDB
# ChromaDB: local, persistent, no API key, simple Python API
# Better for prototyping; Pinecone / Weaviate for production scale
# ----------------------------------------------------------------
import chromadb
from chromadb.utils import embedding_functions

# Persistent local client (data saved to ./chroma_db/)
client = chromadb.PersistentClient(path="./chroma_db")

# Use sentence-transformers embedding function (no API key needed)
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create or load collection
collection_name = "rag_knowledge_base"
try:
    client.delete_collection(collection_name)  # clean run
except:
    pass

collection = client.create_collection(
    name=collection_name,
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"}  # use cosine distance
)

# Add all chunks with metadata
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{"chunk_index": i, "char_count": len(c)} for i, c in enumerate(chunks)]
)

print(f"ChromaDB collection: '{collection_name}'")
print(f"Documents stored: {collection.count()}")

# Query
print("\nCHROMADB RETRIEVAL:")
print("=" * 60)

chroma_queries = [
    "how does ChromaDB differ from FAISS",
    "what embedding model should I use for RAG",
]

for query in chroma_queries:
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["documents", "distances", "metadatas"]
    )
    print(f"\nQuery: '{query}'")
    for doc, dist, meta in zip(
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0]
    ):
        sim = 1 - dist  # ChromaDB returns cosine distance, not similarity
        print(f"  [sim={sim:.4f}] {doc[:70].strip()}")
        print(f"            chunk_index={meta['chunk_index']}, chars={meta['char_count']}")

In [ ]:
# ----------------------------------------------------------------
# Section 6: Full LangChain RAG pipeline
# This is the production pattern from langchain-ai/rag-from-scratch
# LCEL chain: retriever | format | prompt | llm | parser
# ----------------------------------------------------------------
from langchain_community.vectorstores import FAISS as LangFAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# Build LangChain vector store from our knowledge base
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Use RecursiveCharacterTextSplitter for production-quality chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

lc_docs = splitter.create_documents([knowledge_base])
print(f"RecursiveCharacterTextSplitter: {len(lc_docs)} chunks")
print(f"Chunk sizes: {[len(d.page_content) for d in lc_docs]}")

# Build FAISS vector store through LangChain
vectorstore = LangFAISS.from_documents(lc_docs, hf_embeddings)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# RAG prompt template
rag_template = """
Answer the question using only the context provided below.
If the answer is not in the context, say "Not covered in the provided context."

Context:
{context}

Question: {question}
Answer:"""

rag_prompt = PromptTemplate.from_template(rag_template)

def format_docs(docs):
    return "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))

print("\nLangChain LCEL RAG chain built:")
print("  retriever | format_docs -> context")
print("  RunnablePassthrough()   -> question")
print("  rag_prompt | llm | StrOutputParser")

# Test retrieval step only (no LLM needed to verify retrieval works)
test_query = "What is FAISS and what algorithm makes it fast?"
retrieved_docs = retriever.invoke(test_query)
print(f"\nRetrieval test: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"  [{i+1}] {doc.page_content[:70].strip()}...")

# Full chain (requires OPENAI_API_KEY)
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    answer = rag_chain.invoke(test_query)
    print(f"\nLLM Answer: {answer}")
else:
    print("\n[OpenAI API key not set — retrieval is verified, add key to get LLM answer]")

## Real World Problem: RAG Returning Wrong Chunks

A fintech company builds a RAG system to answer questions about their loan products. The system scores 91% on their test set. After deployment, users complain that answers about variable-rate loans mix in details from fixed-rate loan documents.

**Root cause:** The embedding model treats "variable-rate" and "fixed-rate" as similar because they frequently appear in the same context (both are types of loans). The retrieval step returns both types of chunks for almost any loan query.

**Three fixes engineers actually use:**

1. **Metadata filtering:** Tag every chunk with its source document category (`loan_type: variable`, `loan_type: fixed`). At query time, filter by category before running similarity search. ChromaDB and Pinecone both support this with `where` clauses.

2. **Re-ranking:** After retrieving the top-20 chunks by embedding similarity, run a cross-encoder re-ranker on (query, chunk) pairs. Cross-encoders are slower but more precise. Cohere's Rerank API and HuggingFace's `cross-encoder/ms-marco-MiniLM-L-6-v2` both work in production.

3. **Query rewriting:** Before embedding the query, use an LLM to rewrite it into a more specific form. "Tell me about loan interest" becomes "What are the interest rate calculation methods for variable-rate personal loans?" This single step improved retrieval precision by 18% in one documented case.

The bigger lesson: a 91% test set score means almost nothing for RAG systems if the test set doesn't reflect the distribution of actual production queries. Always evaluate RAG systems on real user queries, not synthetic ones.

## Interview Corner: MNC-Level Questions

---

**Q1: Why does RAG outperform fine-tuning for knowledge-intensive tasks when the knowledge changes frequently?**

*What they're testing:* Your understanding of parametric vs non-parametric memory.

*Answer direction:* Fine-tuning bakes knowledge into model weights. Updating that knowledge requires retraining, which is slow, expensive, and has a risk of catastrophic forgetting. RAG stores knowledge externally in a vector database. Updating the knowledge base means re-embedding changed documents, which takes minutes. For a company updating their product documentation weekly, RAG is the only viable option. Fine-tuning makes sense when you need the model to behave differently, not just to know different facts.

---

**Q2: A user asks your RAG system a question but the answer requires combining information from three different chunks that were not retrieved. What's wrong and how do you fix it?**

*What they're testing:* Multi-hop retrieval awareness.

*Answer direction:* Single-round retrieval finds the chunks most similar to the query embedding. If the answer requires synthesising three separate pieces of information that don't individually look similar to the query, standard retrieval fails. Fixes: increase top-k (retrieve more chunks, accept more noise in the context), use a multi-hop retrieval approach (retrieve once, use the first result to generate a follow-up query, retrieve again), or implement a graph-based retrieval that can follow relationships between chunks. Microsoft's GraphRAG paper (2024) specifically addresses this by building a knowledge graph over chunks.

---

**Q3: What chunk size should you use and how do you decide?**

*What they're testing:* Whether you know chunking is a tunable parameter, not a constant.

*Answer direction:* It depends on three things. Your embedding model's context window: all-MiniLM-L6-v2 handles up to 256 tokens, so chunks much larger than that will be truncated. The granularity of your retrieval task: question-answering retrieval benefits from small precise chunks (150-250 tokens); document summarisation benefits from larger chunks (500-1000 tokens). The LLM's context window: if you retrieve 5 chunks of 500 tokens each, you need 2,500 tokens of context plus the prompt plus the answer to fit within the model's limit. In practice, run ablation experiments: test chunk sizes of 128, 256, 512 tokens and measure retrieval precision on a validation query set.

---

**Q4: You embed 100,000 documents with OpenAI's embedding model and store them in a vector database. The model is deprecated. What do you do?**

*What they're testing:* Production embedding lifecycle thinking.

*Answer direction:* You can't use embeddings from one model to search with another. They live in different vector spaces. You need to re-embed all 100,000 documents with the new model and rebuild the index. For OpenAI text-embedding-3-small, 100K documents of ~500 tokens each costs around $1. The real problem is operational: you need to re-embed without downtime. Solution: dual-write during migration (write new embeddings to a new index while the old one stays live), validate the new index on a sample of queries, then cut over traffic. Always store your raw source documents separately from embeddings so you can re-embed whenever needed.

---

**Q5: Explain the difference between IndexFlatIP and IndexIVFFlat in FAISS. When would you use each?**

*What they're testing:* Vector search depth.

*Answer direction:* IndexFlatIP does exact brute-force search: it compares the query against every single vector in the index. Guaranteed to find the true nearest neighbours. Scales as O(n) search time. Use it for up to roughly 100,000 vectors where sub-millisecond latency isn't required. IndexIVFFlat partitions the vector space into nlist clusters (inverted file lists). At search time, it only searches the nprobe closest clusters, not all vectors. This makes search O(nprobe * cluster_size) instead of O(n). For 10M vectors with nlist=1024 and nprobe=16, it searches roughly 1.6% of the index per query. Trade-off: slight recall loss (may miss some true nearest neighbours that fell into an unsearched cluster). Use IndexIVFFlat when your index has more than 100K vectors and you need consistent sub-10ms latency.

## ML Spotlight

**GraphRAG (Microsoft Research, 2024)**

Standard RAG retrieves chunks based on their similarity to the query. If answering a question requires combining facts from across multiple documents, standard RAG fails. The chunks that contain each individual fact look too generic to rank highly for the specific query.

GraphRAG builds a knowledge graph over the document corpus: entities, relationships, and communities of related information. At query time, it traverses this graph rather than searching a flat embedding index. A question about "the relationship between FAISS and approximate nearest neighbour search" can now find and combine all relevant nodes.

On Microsoft's benchmarks, GraphRAG answered complex multi-hop questions that standard RAG could not answer at all, while maintaining comparable accuracy on simpler queries.

It's available as an open-source Python library:
```bash
pip install graphrag
```

Paper: [From Local to Global: A Graph RAG Approach to Query-Focused Summarization](https://arxiv.org/abs/2404.16130)

GitHub: https://github.com/microsoft/graphrag

## Practice Exercise

**Task 1:** Implement overlap chunking and measure whether it improves retrieval accuracy. Compare chunk_fixed with overlap=0 vs overlap=50 vs overlap=100 on these queries against the knowledge base:
- "What is the connection between chunking and retrieval quality?"
- "Does chunk size affect embedding truncation?"

Count how many relevant chunks each strategy retrieves in the top-3.

**Task 2:** Build a RAG system over a real PDF. Use LangChain's `PyPDFLoader`:
```python
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("your_file.pdf")
pages = loader.load()
```
Try it on any technical PDF (a research paper, a product manual). What happens when you ask questions that span multiple pages?

**Task 3:** Implement basic hybrid search. For a given query, get the top-10 results from TF-IDF and the top-10 from FAISS. Combine them by averaging scores, then return the top-3 of the combined set. Does the combined ranking improve on either method alone?

---

**What's Next**

Day 36: LLM Agents — the ReAct loop, tool use, and how agents decide when to search vs reason. Build an agent with three tools from scratch using HuggingFace's smolagents.